In [4]:
%%capture
!pip install -U lightautoml

In [5]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import AdaBoostRegressor

In [6]:
import torch
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task
from lightautoml.report.report_deco import ReportDeco

/opt/conda/lib/python3.10/site-packages/lightautoml/ml_algo/dl_model.py:42: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")


RuntimeError: 

relu(Tensor input, int dim) -> Tensor:
Argument dim not provided.
:
  File "/opt/conda/lib/python3.10/site-packages/lightautoml/ml_algo/torch_based/node_nn_model.py", line 262
    def _forward(input):
        input, is_pos = abs(input), input >= 0
        tau = (input + torch.sqrt(F.relu(8 - input ** 2))) / 2
                                  ~~~~~~ <--- HERE
        tau.masked_fill_(tau <= input, 2.0)
        y_neg = 0.25 * F.relu(tau - input, inplace=True) ** 2


In [ ]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)

In [ ]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['efs_time','ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [ ]:
test.columns, train.columns

In [ ]:
task = Task('reg')

lgb_params = {
    'metric': 'RMSE',
    'lambda_l1': 1e-07, 
    'lambda_l2': 2e-07, 
    'num_leaves': 42, 
    'feature_fraction': 0.55, 
    'bagging_fraction': 0.9, 
    'bagging_freq': 3, 
    'min_child_samples': 19,
    'num_threads': 4
}

cb_params = {
    'num_trees': 10000, 
    'od_wait': 1200, 
    'learning_rate': 0.2, 
    'l2_leaf_reg': 64, 
    'subsample': 0.83, 
    'random_strength': 17.17,  
    'min_data_in_leaf': 10, 
    'leaf_estimation_iterations': 3,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'bootstrap_type': 'Bernoulli',
    'leaf_estimation_method': 'Newton',
    'random_seed': 42,
    "thread_count": 4
}

In [ ]:
automl = TabularAutoML(task = task, 
                       timeout = TIMEOUT,
                       cpu_limit = N_THREADS,
                       reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE},
                       general_params = {'use_algos': [['lgb', 'cb', "lgb_tuned"]]}, # LGBM and CatBoost algos only
                       lgb_params = {'default_params': lgb_params, 'freeze_defaults': True}, # LGBM params
                       cb_params = {'default_params': cb_params, 'freeze_defaults': True}, # CatBoost params
                      )

RD = ReportDeco(output_path = 'tabularAutoML_model_report')
automl_rd = RD(automl)

out_of_fold_predictions = automl_rd.fit_predict(
    train,
    roles = {
        'target': 'efs',
        #'drop': ['unique_id']
        #'weights': 'weight'
    }, 
    verbose = 2
)

In [ ]:
joblib.dump(automl, 'model.pkl')

automl = joblib.load('/kaggle/input/cibmtr-lightautoml-model/model.pkl')

In [ ]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = automl.predict(test)

In [ ]:
submission = pandas.DataFrame({
    'id': id.values,
    'sales_hat': test_predictions.data.reshape(-1),
})
submission

In [ ]:
submission.to_csv('submission.csv', index = False)